# S04 — Backpropagation I

**Week 2 · Wed Sep 2, 2026 · Module 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s04_backpropagation_i.ipynb)

Every cell below is a worked example from the [S04 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s04/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s04.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s04.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## A forward and backward pass by hand


*Expected output starts with:* `forward:  z = 1.0000, h = 0.7616, yhat = -0.6424, L = 1.3487`


In [ ]:
import numpy as np

np.random.seed(0)

# A tiny network on one example:
#   z = w1 * x, h = tanh(z), yhat = w2 * h + b2, L = 0.5 * (yhat - y)^2
x, y = 2.0, 1.0
w1, w2, b2 = 0.5, -1.5, 0.5

# ---- forward pass: compute and store every intermediate ----
z = w1 * x
h = np.tanh(z)
yhat = w2 * h + b2
L = 0.5 * (yhat - y) ** 2
print(f"forward:  z = {z:.4f}, h = {h:.4f}, yhat = {yhat:.4f}, L = {L:.4f}")

# ---- backward pass: walk the graph in reverse, one local derivative each ----
dL_dyhat = yhat - y                    # d/dyhat of 0.5*(yhat - y)^2
dL_db2 = dL_dyhat * 1.0                # yhat = w2*h + b2  ->  dyhat/db2 = 1
dL_dw2 = dL_dyhat * h                  #                      dyhat/dw2 = h
dL_dh = dL_dyhat * w2                  #                      dyhat/dh  = w2
dL_dz = dL_dh * (1 - h ** 2)           # h = tanh(z)       ->  dh/dz = 1 - tanh(z)^2
dL_dw1 = dL_dz * x                     # z = w1*x          ->  dz/dw1 = x
print(f"backward: dL/dyhat = {dL_dyhat:.4f}, dL/dw2 = {dL_dw2:.4f}, "
      f"dL/db2 = {dL_db2:.4f}, dL/dw1 = {dL_dw1:.4f}")

# ---- check dL/dw1 against a finite difference ----
def loss_at(w1_val):
    return 0.5 * (w2 * np.tanh(w1_val * x) + b2 - y) ** 2

eps = 1e-6
fd = (loss_at(w1 + eps) - loss_at(w1 - eps)) / (2 * eps)
print(f"finite difference for dL/dw1: {fd:.4f}")

## Forward mode, in code


*Expected output starts with:* `pass seeded at w1: L = 1.3487, dL/dw1 = 2.0693`


In [ ]:
import numpy as np

np.random.seed(0)

class Dual:
    """A value paired with a derivative, propagated forward through every op."""
    def __init__(self, val, dot):
        self.val, self.dot = val, dot
    def _wrap(self, o):
        return o if isinstance(o, Dual) else Dual(o, 0.0)
    def __add__(self, o):
        o = self._wrap(o); return Dual(self.val + o.val, self.dot + o.dot)
    __radd__ = __add__
    def __sub__(self, o):
        o = self._wrap(o); return Dual(self.val - o.val, self.dot - o.dot)
    def __mul__(self, o):
        o = self._wrap(o)
        return Dual(self.val * o.val, self.dot * o.val + self.val * o.dot)
    __rmul__ = __mul__
    def __pow__(self, k):
        return Dual(self.val ** k, k * self.val ** (k - 1) * self.dot)

def tanh(a):
    t = np.tanh(a.val)
    return Dual(t, (1 - t ** 2) * a.dot)

# The same tiny network as the by-hand example
x, y = 2.0, 1.0
params = {"w1": 0.5, "w2": -1.5, "b2": 0.5}

def loss(w1, w2, b2):
    z = w1 * x
    h = tanh(z)
    yhat = w2 * h + b2
    return 0.5 * (yhat - y) ** 2

# Forward mode: ONE pass per parameter, seeding that parameter's dot with 1
for name in params:
    args = {k: Dual(v, 1.0 if k == name else 0.0) for k, v in params.items()}
    L = loss(**args)
    print(f"pass seeded at {name}: L = {L.val:.4f}, dL/d{name} = {L.dot:.4f}")

## Vectorized backprop: whole layers at a time


*Expected output starts with:* `loss = 1.0065`


In [ ]:
import numpy as np

np.random.seed(0)

# Batch of 8 examples, 3 features, 4 hidden ReLU units, 2 output classes
N, D, H, C = 8, 3, 4, 2
X = np.random.randn(N, D)
y = np.random.randint(0, C, size=N)

params = {
    "W1": 0.5 * np.random.randn(D, H), "b1": np.zeros(H),
    "W2": 0.5 * np.random.randn(H, C), "b2": np.zeros(C),
}

def forward_backward(params, X, y):
    W1, b1, W2, b2 = params["W1"], params["b1"], params["W2"], params["b2"]
    N = X.shape[0]

    # forward
    Z1 = X @ W1 + b1                         # (N, H) pre-activations
    A1 = np.maximum(0, Z1)                   # (N, H) ReLU
    Z2 = A1 @ W2 + b2                        # (N, C) logits
    Z2s = Z2 - Z2.max(axis=1, keepdims=True)             # stable log-softmax
    logp = Z2s - np.log(np.exp(Z2s).sum(axis=1, keepdims=True))
    loss = -logp[np.arange(N), y].mean()

    # backward — same graph, reverse order
    dZ2 = np.exp(logp)                       # softmax probabilities, (N, C)
    dZ2[np.arange(N), y] -= 1                # subtract 1 at the true class
    dZ2 /= N                                 # mean over the batch
    grads = {
        "W2": A1.T @ dZ2, "b2": dZ2.sum(axis=0),
    }
    dA1 = dZ2 @ W2.T                         # (N, H)
    dZ1 = dA1 * (Z1 > 0)                     # ReLU gate: pass or block
    grads["W1"] = X.T @ dZ1
    grads["b1"] = dZ1.sum(axis=0)
    return loss, grads

loss, grads = forward_backward(params, X, y)
print(f"loss = {loss:.4f}")

# Gradient check: compare every analytic gradient to a central finite difference
eps = 1e-5
for name, p in params.items():
    num = np.zeros_like(p)
    for idx in np.ndindex(p.shape):
        old = p[idx]
        p[idx] = old + eps; lp, _ = forward_backward(params, X, y)
        p[idx] = old - eps; lm, _ = forward_backward(params, X, y)
        p[idx] = old
        num[idx] = (lp - lm) / (2 * eps)
    rel = np.abs(num - grads[name]) / (np.abs(num) + np.abs(grads[name]) + 1e-12)
    print(f"{name}: max relative error vs finite difference = {rel.max():.2e}")

## Gradient checking as a habit


*Expected output starts with:* `float64, eps = 1e-02: max relative error = 3.10e-05`


In [ ]:
import numpy as np

np.random.seed(0)

def loss_and_grad(w, X, y):
    """Logistic regression loss and analytic gradient, in w's dtype."""
    p = 1 / (1 + np.exp(-(X @ w)))
    tiny = np.finfo(w.dtype).tiny
    L = -np.mean(y * np.log(p + tiny) + (1 - y) * np.log(1 - p + tiny))
    g = X.T @ (p - y) / len(y)
    return L, g

for dtype in [np.float64, np.float32]:
    X = np.random.default_rng(0).standard_normal((64, 10)).astype(dtype)
    w = (0.1 * np.random.default_rng(1).standard_normal(10)).astype(dtype)
    yv = (np.random.default_rng(2).uniform(size=64) > 0.5).astype(dtype)
    _, g = loss_and_grad(w, X, yv)
    for eps in [1e-2, 1e-4, 1e-6]:
        num = np.zeros_like(g)
        for i in range(len(w)):
            wp = w.copy(); wp[i] += dtype(eps)
            wm = w.copy(); wm[i] -= dtype(eps)
            num[i] = (loss_and_grad(wp, X, yv)[0] - loss_and_grad(wm, X, yv)[0]) / (2 * eps)
        rel = np.max(np.abs(num - g) / (np.abs(num) + np.abs(g) + 1e-12))
        print(f"{np.dtype(dtype).name}, eps = {eps:.0e}: max relative error = {rel:.2e}")

## Going deeper


*Expected output starts with:* `activations stored, standard backprop:  17`


In [ ]:
import numpy as np

np.random.seed(0)

# A chain of 16 tanh layers; loss = mean of the final output.
L_layers, width = 16, 32
Ws = [np.random.randn(width, width) / np.sqrt(width) for _ in range(L_layers)]
x0 = np.random.randn(8, width)

def backward_full(x):
    """Standard backprop: store every activation, then sweep backward."""
    acts = [x]
    for W in Ws:
        acts.append(np.tanh(acts[-1] @ W))
    d = np.ones_like(acts[-1]) / acts[-1].size          # dL/d(output) for L = mean
    grads = [None] * L_layers
    for i in reversed(range(L_layers)):
        dz = d * (1 - acts[i + 1] ** 2)                 # through tanh
        grads[i] = acts[i].T @ dz
        d = dz @ Ws[i].T
    return grads, len(acts)

def backward_checkpointed(x, every=4):
    """Keep every 4th activation; recompute each segment during the backward pass."""
    ckpts, h = {0: x}, x
    for i, W in enumerate(Ws):
        h = np.tanh(h @ W)
        if (i + 1) % every == 0:
            ckpts[i + 1] = h
    d = np.ones_like(h) / h.size
    grads = [None] * L_layers
    for seg_end in range(L_layers, 0, -every):
        seg_start = seg_end - every
        seg = [ckpts[seg_start]]                        # recompute this segment only
        for i in range(seg_start, seg_end):
            seg.append(np.tanh(seg[-1] @ Ws[i]))
        for i in reversed(range(seg_start, seg_end)):
            dz = d * (1 - seg[i - seg_start + 1] ** 2)
            grads[i] = seg[i - seg_start].T @ dz
            d = dz @ Ws[i].T
    return grads, len(ckpts) + every + 1                # checkpoints + one live segment

g_full, mem_full = backward_full(x0)
g_ck, mem_ck = backward_checkpointed(x0)
diff = max(np.max(np.abs(a - b)) for a, b in zip(g_full, g_ck))
print(f"activations stored, standard backprop:  {mem_full}")
print(f"activations stored, checkpointed:       {mem_ck} (5 checkpoints + 1 segment of 4 + input)")
print(f"max |gradient difference|:              {diff:.2e}")

## Try it yourself

1. Extend the by-hand example with a first-layer bias `b1` (so `z = w1 * x + b1`). Derive `dL/db1` on paper, add it to the script, and verify it with a finite difference.
2. Add a `sin` operation to the `Dual` class (values `sin(v)`, derivative `cos(v) * dot`), change the tiny network's activation from `tanh` to `sin`, and verify all three forward-mode gradients against finite differences.
3. In the vectorized example, deliberately break the backward pass three ways — remove the ReLU mask `(Z1 > 0)`, drop the `/= N`, transpose `W2.T` to `W2` — and record the gradient check's report for each. Learn what each class of bug "looks like."
4. Add a second hidden layer (shapes `3 -> 4 -> 4 -> 2`) to the vectorized network and make the gradient check pass again. How many lines did the backward pass grow by, and which lines are pure repetition?
5. In the checkpointing example, vary `every` over 1, 2, 4, 8, 16 and tabulate stored activations against the number of extra forward-layer computations. Which setting minimizes total storage for this 16-layer chain, and how does that compare to `sqrt(16)`?
6. Measure the cost claim: time 1000 forward passes and 1000 forward+backward passes of the vectorized network (increase sizes to `N, D, H, C = 256, 100, 100, 10` so timings are stable). Is forward+backward within a small factor of twice the forward-only time?


---

Full discussion of everything above: [S04 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s04/).
